In [ ]:
"""
This script tests samples periodically from a bioreactor. 
The samples are dispensed into a BioER Deepwell plate containing buffer like arresting buffer
"""


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:

import asyncio
from datetime import datetime, timedelta

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import (
    Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint,  # module for pioreactor
    Hamilton_MFX_plateholder_DWP_metal_tapped                 # module for DW plate
)
from pylabrobot.resources.bioer import BioER_96_wellplate_Vb_2200ul
from pylabrobot.resources.pioreactor import pioreactor
from pylabrobot.resources import (
    TIP_50ul_w_filter,  # 50 µL filtered
    HTF,                # 1000 µL filtered
    LTF                 # 10 µL filtered
)
from pylabrobot.resources import Coordinate

###############################################################################
# 0) Build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)

In [ ]:

###############################################################################
# 1) Carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)

tiprack_1000 = HTF("tips_00")               # 1000 µL filter tips (slot-0)
tiprack_50   = TIP_50ul_w_filter("tips_01")  #  50 µL filter tips (slot-1)
tiprack_10   = LTF("tips_02")                #  10 µL filter tips (slot-2)

# Mount the racks
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10

# --- deep-well plate on MFX holder ------------------------------------------
module0 = Hamilton_MFX_plateholder_DWP_metal_tapped("module0")
car_13 = MFX_CAR_L5_base("car_13", modules={0: module0})
lh.deck.assign_child_resource(car_13, rails=13)

dwplate = BioER_96_wellplate_Vb_2200ul("dwPlate")
module0.assign_child_resource(dwplate)

# --- pioreactor on the 10mm raised holder -----------------------------------
moduleMod = Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("moduleMod")
car_07 = MFX_CAR_L5_base("car_07", modules={0: moduleMod})
lh.deck.assign_child_resource(car_07, rails=7)

pr = pioreactor("pr")
moduleMod.assign_child_resource(pr)

In [20]:
###############################################################################
# 1a) Pioreactor Stirring--Starting and Stopping
###############################################################################
import time
import requests
from urllib.parse import quote

BASE = "http://leadera1.local"
UNIT = "leaderA1"
EXP  = "sample_and_aliquoting"

def run_stirring(rpm=500):
    url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(url, json={"options": {"target_rpm": rpm}},
                          headers={"Content-Type": "application/json"})
    print("START:", resp.status_code, resp.text)
    return resp

def update_stirring(rpm):
    url = f"{BASE}/api/workers/{UNIT}/jobs/update/job_name/stirring/experiments/{quote(EXP)}"
    # NOTE: update uses "settings", not "options"
    resp = requests.patch(url, json={"settings": {"target_rpm": rpm}},
                          headers={"Content-Type": "application/json"})
    print("UPDATE:", resp.status_code, resp.text)
    return resp

def list_running():
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    r = requests.get(url)
    try:
        txt = r.text
    except Exception:
        txt = ""
    print("RUNNING:", r.status_code, txt)
    return r

def wait_until_stopped(job_name="stirring", timeout_s=15, poll_s=0.5):
    """Poll /jobs/running until job_name disappears or timeout."""
    deadline = time.time() + timeout_s
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    while time.time() < deadline:
        r = requests.get(url)
        if r.ok:
            if job_name not in r.text:
                return True
        time.sleep(poll_s)
    return False

def stop_stirring():
    # Preferred “stop this specific job” endpoint
    stop_specific = f"{BASE}/api/workers/{UNIT}/jobs/stop/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(stop_specific, headers={"Content-Type": "application/json"})
    print("STOP specific:", resp.status_code, getattr(resp, "text", ""))

    if wait_until_stopped():
        print("Confirmed: stirring stopped.")
        return True

    # Fallback #1: stop ALL jobs for this unit in this experiment
    stop_all = f"{BASE}/api/workers/{UNIT}/jobs/stop/experiments/{quote(EXP)}"
    resp2 = requests.patch(stop_all, headers={"Content-Type": "application/json"})
    print("STOP all-in-exp:", resp2.status_code, getattr(resp2, "text", ""))

    if wait_until_stopped():
        print("Confirmed after stop-all: stirring stopped.")
        return True

    # Fallback #2: unit_api stop with query params (works across versions)
    # (Some releases deprecate path-style unit_api stops in favour of query params.)
    unit_api_stop = f"{BASE}/unit_api/jobs/stop?job_name=stirring&experiment={quote(EXP)}"
    resp3 = requests.patch(unit_api_stop, headers={"Content-Type": "application/json"})
    print("STOP unit_api:", resp3.status_code, getattr(resp3, "text", ""))

    ok = wait_until_stopped()
    print("Final stop status:", "stopped" if ok else "still running")
    return ok

# if __name__ == "__main__":
#     # 1) start
#     run_stirring(600)

#     # 2) verify running
#     list_running()

#     # 3) update RPM correctly (uses "settings")
#     update_stirring(400)
#     time.sleep(10)
#     # 4) stop sequence with verification + fallbacks
#     stopped = stop_stirring()
#     if not stopped:
#         # Optional: drop RPM to 0 first, then re-try stop
#         update_stirring(0)
#         print("Retrying stop after setting RPM=0 …")
#         stop_stirring()


In [21]:

###############################################################################
# 2) Parameters
###############################################################################
CYCLES = 11
TIME_BETWEEN_SAMPLING_MIN = 30  # minutes
CHANNEL_MM = 6                  # use channel 6 for the run
TIPRACK_50 = tiprack_50         # convenience alias
SAFE_Z = 270.0                  # mm above deck; should exceed tallest stack

###############################################################################
# 3) Helpers
###############################################################################
async def countdown(seconds: int, prefix: str = ""):
  """Simple terminal countdown (updates ~1/s)."""
  end = datetime.now() + timedelta(seconds=seconds)
  while True:
    remaining = int((end - datetime.now()).total_seconds())
    if remaining <= 0:
      print(f"\r{prefix}00:00 remaining. Starting next cycle...       ")
      break
    mins, secs = divmod(remaining, 60)
    print(f"\r{prefix}{mins:02d}:{secs:02d} remaining (ETA {end.strftime('%H:%M:%S')})   ", end="")
    await asyncio.sleep(1)
  # Move to next line after countdown
  print()

###############################################################################
# 4) Core action
###############################################################################
async def sample_and_aliquot(col: int):
  """Pick up tip, aspirate from pioreactor, dispense into two wells, discard tip."""
  try:
    # Tips: row A, moving across columns
    await lh.pick_up_tips(TIPRACK_50[f"A{col}"], use_channels=[CHANNEL_MM])
    
    # Aspirate from pioreactor (hover above crossbar a bit)
    await lh.aspirate(
      pr["A1"],
      vols=[45],
      use_channels=[CHANNEL_MM],
      transport_air_volume=[0],
      # pre_wetting_volume=[45], # this doesn't work with 50ul filtered tips
      mix_volume = [50],
      mix_cycles = [2],
      settling_time=[2],
      liquid_height = [8], # above crossbar
      # offsets=[Coordinate(z=8.0)],  # ~8 mm above crossbar
      minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
      min_z_endpos=SAFE_Z,
    )

    # Dispense 20 µL into A{col}
    await lh.dispense(
      dwplate[f"A{col}"],
      vols=[20],
      use_channels=[CHANNEL_MM],
      liquid_height=[20],  # ~1 mL buffer preloaded in DW well
      settling_time=[1],
      transport_air_volume=[0],
      minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
      min_z_endpos=SAFE_Z,
    )

    # Dispense 20 µL into E{col}
    await lh.dispense(
      dwplate[f"E{col}"],
      vols=[20],
      use_channels=[CHANNEL_MM],
      liquid_height=[20],  # ~1 mL buffer preloaded
      settling_time=[1],
      transport_air_volume=[0],
      minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
      min_z_endpos=SAFE_Z,
    )
  finally:
    # Always try to discard tips to keep state sane
    try:
      await lh.discard_tips(use_channels=[CHANNEL_MM])
    except Exception:
      pass



In [22]:
###############################################################################
# 5) Run cycles with status UI inside the loop
###############################################################################
start_time = datetime.now()
overall_eta = start_time + timedelta(minutes=TIME_BETWEEN_SAMPLING_MIN * (CYCLES - 1))
print(f"Run started at {start_time.strftime('%H:%M:%S')}. Estimated completion ~ {overall_eta.strftime('%H:%M:%S')}.")

for c in range(1, CYCLES + 1):
  cycle_start = datetime.now()
  # === Status line: current cycle / total, begin time, expected end time ===
  exp_end = cycle_start + timedelta(minutes=TIME_BETWEEN_SAMPLING_MIN) if c < CYCLES else None
  print(f"\nCycle {c}/{CYCLES} — begin {cycle_start.strftime('%H:%M:%S')}"
        + (f", next cycle ETA {exp_end.strftime('%H:%M:%S')}" if exp_end else ", final cycle"))

  stopped = stop_stirring()
  time.sleep(5)
  if not stopped:
      # Optional: drop RPM to 0 first, then re-try stop
      update_stirring(0)
      time.sleep(5)
      print("Retrying stop after setting RPM=0 …")
      stop_stirring()
# Use columns 1..12 then wrap if needed (simple example)
  time.sleep(10) # chill before aliquoting
  col = ((c - 1) % 12) + 1
  await sample_and_aliquot(col)
  run_stirring(600)

  # === Countdown visualizer (between cycles only) ===
  if c < CYCLES:
    wait_seconds = int(TIME_BETWEEN_SAMPLING_MIN * 60)
    await countdown(wait_seconds, prefix=f"Waiting to start cycle {c+1}/{CYCLES}: ")

Run started at 14:19:46. Estimated completion ~ 19:19:46.

Cycle 1/11 — begin 14:19:46, next cycle ETA 14:49:46
STOP specific: 202 {"status":"success"}
Confirmed: stirring stopped.
START: 202 {"unit":"leaderA1","task_id":"ab0d0999-0b5b-4500-9e12-f616ef581bf6","result_url_path":"/unit_api/task_results/ab0d0999-0b5b-4500-9e12-f616ef581bf6"}
Waiting to start cycle 2/11: 00:00 remaining. Starting next cycle...       


Cycle 2/11 — begin 14:50:37, next cycle ETA 15:20:37
STOP specific: 202 {"status":"success"}
Confirmed: stirring stopped.
START: 202 {"unit":"leaderA1","task_id":"e4e04761-45d9-4d57-b849-10ccb2a89b70","result_url_path":"/unit_api/task_results/e4e04761-45d9-4d57-b849-10ccb2a89b70"}
Waiting to start cycle 3/11: 00:00 remaining. Starting next cycle...       


Cycle 3/11 — begin 15:21:28, next cycle ETA 15:51:28
STOP specific: 202 {"status":"success"}
Confirmed: stirring stopped.
START: 202 {"unit":"leaderA1","task_id":"29e60c30-b686-4731-ac4a-e922a1125fc7","result_url_path":"/

In [ ]:
# await lh.dispense(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
# await lh.dispense(SRC_MM_TUBE, vols=[100], liquid_height=[4], use_channels=[CHANNEL_MM], blow_out=[1])
# await lh.dispense(pr7["A1"], vols=[50], use_channels=[CHANNEL_MM], blow_out=[1])
# await lh.drop_tips(TIPRACK_50["A1"], use_channels=[6])
# await lh.discard_tips()
# await lh.stop()
# await backend.stop() 
# # # Aspirate from pioreactor (hover above crossbar a bit)
# await lh.aspirate(
# pr["A1"],
# vols=[45],
# use_channels=[CHANNEL_MM],
# transport_air_volume=[0],
# # pre_wetting_volume=[45], # this doesn't work with 50ul filtered tips
# mix_volume = [50],
# mix_cycles = [2],
# settling_time=[2],
# liquid_height = [8], # above crossbar
# flow_rates=[20],
# # offsets=[Coordinate(z=8.0)],  # ~8 mm above crossbar
# minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
# min_z_endpos=SAFE_Z,
# )

# # Dispense 20 µL into E{col}
# await lh.dispense(
# dwplate["E1"],
# vols=[45],
# use_channels=[CHANNEL_MM],
# liquid_height=[20],  # ~1 mL buffer preloaded
# settling_time=[1],
# transport_air_volume=[0],
# minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
# min_z_endpos=SAFE_Z,
# )